# 05 · Do balance-sheet fundamentals explain peg deviations?

$$y_{e,t} = a + \rho\,y_{e,t-1} + \beta' f(L_t, K_t) + \gamma\,\text{VIX}_t + \delta_1 \text{HL}_{e,t-1}
           + \delta_2 \ln \text{Vol}_{e,t-1} + \text{FE}_e + \varepsilon_{e,t}$$

* $y$: daily log peg deviation per exchange, winsorised at 0.1%/99.9%
* $L$: liquidity ratio (cash + deposits)/reserves · $K$: equity buffer (reserves − supply)/reserves, from the
  daily mark-to-market reserve reconstruction
* USDC uses a composite safety score $z(L)+z(K)$; USDT enters $z(L)$, $z(K)$ separately
* Stress specifications interact the balance-sheet terms and centred VIX with a top-decile-VIX or SVB-window dummy

The paper reports Newey–West errors on the stacked panel. Because the kernel then runs across exchange
boundaries, Driscoll–Kraay errors (robust to cross-sectional and serial correlation) are reported as a check.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import regressions
from stablecoin_risk.data import loaders

usdc_panel = regressions.build_panel(*loaders.load_regression_inputs("USDC"))
usdc = regressions.run_paper_specs("USDC", panel=usdc_panel)
regressions.results_table(usdc)

,Full,Calm,High-VIX x stress,SVB x stress
const,0.000147***,9.16e-05***,0.000136***,0.000109***
(const),(4.44e-05),(1.78e-05),(3.94e-05),(3.29e-05)
y_lag1,0.532***,0.496***,0.53***,0.459***
(y_lag1),(0.0861),(0.0396),(0.0834),(0.0772)
safety,1.62e-05**,7.14e-06***,3.3e-06**,3.2e-06*
(safety),(8.13e-06),(1.47e-06),(1.67e-06),(1.79e-06)
vix,-1.46e-06,-2.63e-07,,
(vix),(1.31e-06),(6.2e-07),,
hl_lag1,-0.00016,-1.79e-05*,-0.000139,-0.000129
(hl_lag1),(0.000199),(9.21e-06),(0.000184),(0.000147)


In [3]:
regressions.results_table(regressions.run_paper_specs("USDC", cov="driscoll-kraay", panel=usdc_panel))

,Full,Calm,High-VIX x stress,SVB x stress
const,0.000147**,9.16e-05***,0.000136***,0.000109***
(const),(5.74e-05),(2.01e-05),(4.75e-05),(3.05e-05)
y_lag1,0.532***,0.496***,0.53***,0.459***
(y_lag1),(0.0963),(0.037),(0.0921),(0.0828)
safety,1.62e-05,7.14e-06***,3.3e-06*,3.2e-06*
(safety),(1.04e-05),(1.68e-06),(1.79e-06),(1.91e-06)
vix,-1.46e-06,-2.63e-07,,
(vix),(1.79e-06),(7.74e-07),,
hl_lag1,-0.00016,-1.79e-05*,-0.000139,-0.000129
(hl_lag1),(0.000218),(9.37e-06),(0.000196),(0.000161)


In [4]:
usdt_panel = regressions.build_panel(*loaders.load_regression_inputs("USDT"))
regressions.results_table(regressions.run_paper_specs("USDT", panel=usdt_panel))

,Full,Calm,High-VIX x stress,SVB x stress
const,-0.000428,0.000592,-1.44e-05,0.000106
(const),(0.000572),(0.00112),(0.000598),(0.000613)
y_lag1,0.87***,0.845***,0.869***,0.869***
(y_lag1),(0.0807),(0.0808),(0.0806),(0.0809)
zL,-0.000102,-7.53e-05,-0.000111,-0.000111
(zL),(7.79e-05),(0.000107),(8.11e-05),(8.11e-05)
zK,-0.000104,-1.9e-05,-6.57e-05,-0.000104
(zK),(0.000117),(0.000134),(0.00013),(0.00012)
vix,2.38e-05*,1.55e-05,,
(vix),(1.28e-05),(1.22e-05),,


In [5]:
regressions.results_table(regressions.run_paper_specs("USDT", cov="driscoll-kraay", panel=usdt_panel))

,Full,Calm,High-VIX x stress,SVB x stress
const,-0.000428,0.000592,-1.44e-05,0.000106
(const),(0.000597),(0.00113),(0.000611),(0.000617)
y_lag1,0.87***,0.845***,0.869***,0.869***
(y_lag1),(0.0808),(0.081),(0.0808),(0.081)
zL,-0.000102,-7.53e-05,-0.000111,-0.000111
(zL),(7.74e-05),(0.000108),(8.06e-05),(8.02e-05)
zK,-0.000104,-1.9e-05,-6.57e-05,-0.000104
(zK),(0.000116),(0.000134),(0.000128),(0.000119)
vix,2.38e-05*,1.55e-05,,
(vix),(1.36e-05),(1.27e-05),,


**Reading the tables.**

* **USDC.** A stronger balance sheet (higher composite safety) is associated with prices closer to or above par,
  most robustly in the calm sample (significant at 1% under both error types). Peg deviations are highly persistent
  (ρ ≈ 0.5), and lagged trading volume consistently tightens the peg.
* **USDT.** Deviations are even more persistent (ρ ≈ 0.87). In the SVB window, liquidity ($z(L)$) and equity ($z(K)$)
  interactions take *opposite* signs under Newey–West errors, consistent with liquidity and capital being distinct
  channels. **Caveat:** those interactions are not significant under Driscoll–Kraay errors, which allow for
  correlation across exchanges on the same day. With only ~8 SVB-window days per exchange, treat them as suggestive.